In [1]:
import pandas as pd # type: ignore
import numpy as np # type: ignore
import re
from datetime import datetime, timedelta, timezone
from dateutil.relativedelta import relativedelta
import os
import io

In [ ]:
# -----------------------
# Config
# -----------------------
columns_to_read = [
    'Product variant sku', 'Day', 'Ending inventory units',
    'Days in inventory', 'Days in stock', 'Days out of stock' ]
sku_aliases = ['product variant sku', 'sku', 'product sku']

path = r"C:\Users\Ashutosh Srivastava\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands- Client Repository\Client Database\Kinetica\Shopify\ie\inventory_ledger"

df_list = []

# -----------------------
# Current and Last Month
# -----------------------
today = datetime.today()
current_month_num = today.month
last_month_num = current_month_num - 1 if current_month_num > 1 else 12

# -----------------------
# Standardize Day column
# -----------------------
def standardize_day_by_source(day_value, source_type):
    """
    Standardize Day column by forcing month based on source file.
    Returns pd.Timestamp in YYYY-MM-DD
    """
    if pd.isna(day_value):
        return pd.NaT

    # Normalize delimiters
    day_str = str(day_value).strip().replace('/', '-').replace('.', '-').split(' ')[0]

    parts = day_str.split('-')
    if len(parts) != 3:
        return pd.NaT

    # Convert to integers
    try:
        parts = [int(p) for p in parts]
    except:
        return pd.NaT

    # Determine the target month
    target_month = current_month_num if source_type=='current' else last_month_num

    # Try to find month in parts
    month_idx = None
    for i, p in enumerate(parts):
        if p == target_month:
            month_idx = i
            break

    if month_idx is None:
        month_idx = 1  # fallback

    remaining = [p for i, p in enumerate(parts) if i != month_idx]
    day = remaining[0] if remaining[0] <= 31 else remaining[1]
    year = remaining[1] if remaining[0] <= 31 else remaining[0]
    if year < 100:
        year += 2000  # handle 2-digit year

    try:
        return pd.Timestamp(year=year, month=target_month, day=day)
    except:
        return pd.NaT

# -----------------------
# Column Normalizer
# -----------------------
def normalize_columns(df):
    col_map = {}
    for col in df.columns:
        normalized = col.strip().lower()
        if normalized in sku_aliases:
            col_map[col] = 'Product variant sku'
        elif normalized == 'day':
            col_map[col] = 'Day'
        elif normalized == 'ending inventory units':
            col_map[col] = 'Ending inventory units'
        elif normalized == 'days in inventory':
            col_map[col] = 'Days in inventory'
        elif normalized == 'days in stock':
            col_map[col] = 'Days in stock'
        elif normalized == 'days out of stock':
            col_map[col] = 'Days out of stock'
    df = df.rename(columns=col_map)
    return df

# -----------------------
# Load all files
# -----------------------
for root, dirs, files in os.walk(path):
    for file in files:
        file_path = os.path.join(root, file)
        print(f"Processing: {file}")
        try:
            # Read file
            if file.endswith((".xlsx", ".xls")):
                df = pd.read_excel(file_path, dtype=str)
            elif file.endswith(".csv"):
                df = pd.read_csv(file_path, dtype=str, low_memory=False)
            elif file.endswith(".txt"):
                df = pd.read_csv(file_path, delimiter='\t', encoding='utf-8', dtype=str, low_memory=False)
            else:
                continue

            # Normalize columns
            df = normalize_columns(df)

            # Keep only required columns
            available_cols = [col for col in columns_to_read if col in df.columns]
            df = df[available_cols]

            # Standardize Day column based on file type
            if 'Current Month' in file:
                df['Day'] = df['Day'].apply(lambda x: standardize_day_by_source(x, 'current'))
            elif 'Last Month' in file:
                df['Day'] = df['Day'].apply(lambda x: standardize_day_by_source(x, 'last'))

            df['Source_File'] = file  # keep source info if needed
            df_list.append(df)
            print(f"✅ Successfully read ledger file: {file} ({len(df)} rows)")
        except Exception as e:
            print(f"❌ Skipping {file} due to error: {e}")
            print(f"   Full path: {file_path}")
# -----------------------
# Combine all
# -----------------------
if df_list:
    Shopify_ledger_overall = pd.concat(df_list, ignore_index=True)
else:
    Shopify_ledger_overall = pd.DataFrame(columns=columns_to_read)

# -----------------------
# Clean and Format
# -----------------------
if not Shopify_ledger_overall.empty:
    Shopify_ledger_overall = Shopify_ledger_overall[~Shopify_ledger_overall['Product variant sku'].isnull()]
    Shopify_ledger_overall['country'] = "IE"
    Shopify_ledger_overall['platform'] = "Shopify"

    Shopify_ledger_overall.rename(
        columns={
            'Product variant sku': 'SKU',
            'Day': 'Date',
            'Ending inventory units': 'Ending_Warehouse_Balance'},
        inplace=True)

    # SKU cleanup
    Shopify_ledger_overall['SKU'] = 'KCA-' + Shopify_ledger_overall['SKU'].astype(str).str.replace(r'\.0$', '', regex=True)
    Shopify_ledger_overall = Shopify_ledger_overall[Shopify_ledger_overall['SKU'] != 'KCA-nan']

    # Create Year-Month for aggregation
    Shopify_ledger_overall['Year-Month'] = Shopify_ledger_overall['Date'].dt.strftime('%Y-%m')

    # Convert Ending_Warehouse_Balance to numeric
    Shopify_ledger_overall['Ending_Warehouse_Balance'] = pd.to_numeric(Shopify_ledger_overall['Ending_Warehouse_Balance'], errors='coerce').fillna(0)

# for UK

In [ ]:
columns_to_read = [
        'Product variant sku', 'Day', 'Ending inventory units',
        'Days in inventory', 'Days in stock', 'Days out of stock' ]
    sku_aliases = ['product variant sku', 'sku', 'product sku']

    path = r"C:\Users\Ashutosh Srivastava\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands- Client Repository\Client Database\Kinetica\Shopify\uk\inventory_ledger"

    df_list = []

    # -----------------------
    # Current and Last Month
    # -----------------------
    today = datetime.today()
    current_month_num = today.month
    last_month_num = current_month_num - 1 if current_month_num > 1 else 12

    # -----------------------
    # Standardize Day column
    # -----------------------
    def standardize_day_by_source(day_value, source_type):
        """
        Standardize Day column by forcing month based on source file.
        Returns pd.Timestamp in YYYY-MM-DD
        """
        if pd.isna(day_value):
            return pd.NaT

        # Normalize delimiters
        day_str = str(day_value).strip().replace('/', '-').replace('.', '-').split(' ')[0]

        parts = day_str.split('-')
        if len(parts) != 3:
            return pd.NaT

        # Convert to integers
        try:
            parts = [int(p) for p in parts]
        except:
            return pd.NaT

        # Determine the target month
        target_month = current_month_num if source_type=='current' else last_month_num

        # Try to find month in parts
        month_idx = None
        for i, p in enumerate(parts):
            if p == target_month:
                month_idx = i
                break

        if month_idx is None:
            month_idx = 1  # fallback

        remaining = [p for i, p in enumerate(parts) if i != month_idx]
        day = remaining[0] if remaining[0] <= 31 else remaining[1]
        year = remaining[1] if remaining[0] <= 31 else remaining[0]
        if year < 100:
            year += 2000  # handle 2-digit year

        try:
            return pd.Timestamp(year=year, month=target_month, day=day)
        except:
            return pd.NaT

    # -----------------------
    # Column Normalizer
    # -----------------------
    def normalize_columns(df):
        col_map = {}
        for col in df.columns:
            normalized = col.strip().lower()
            if normalized in sku_aliases:
                col_map[col] = 'Product variant sku'
            elif normalized == 'day':
                col_map[col] = 'Day'
            elif normalized == 'ending inventory units':
                col_map[col] = 'Ending inventory units'
            elif normalized == 'days in inventory':
                col_map[col] = 'Days in inventory'
            elif normalized == 'days in stock':
                col_map[col] = 'Days in stock'
            elif normalized == 'days out of stock':
                col_map[col] = 'Days out of stock'
        df = df.rename(columns=col_map)
        return df

    # -----------------------
    # Load all files
    # -----------------------
    for root, dirs, files in os.walk(path):
        for file in files:
            file_path = os.path.join(root, file)
            log_event(f"Processing: {file}")
            try:
                # Read file
                if file.endswith((".xlsx", ".xls")):
                    df = pd.read_excel(file_path, dtype=str)
                elif file.endswith(".csv"):
                    df = pd.read_csv(file_path, dtype=str, low_memory=False)
                elif file.endswith(".txt"):
                    df = pd.read_csv(file_path, delimiter='\t', encoding='utf-8', dtype=str, low_memory=False)
                else:
                    continue

                # Normalize columns
                df = normalize_columns(df)

                # Keep only required columns
                available_cols = [col for col in columns_to_read if col in df.columns]
                df = df[available_cols]

                # Standardize Day column based on file type
                if 'Current Month' in file:
                    df['Day'] = df['Day'].apply(lambda x: standardize_day_by_source(x, 'current'))
                elif 'Last Month' in file:
                    df['Day'] = df['Day'].apply(lambda x: standardize_day_by_source(x, 'last'))

                df['Source_File'] = file  # keep source info if needed
                df_list.append(df)
                log_event(f"✅ Successfully read ledger file: {file} ({len(df)} rows)")
            except Exception as e:
                log_event(f"❌ Skipping {file} due to error: {e}")
                log_event(f"   Full path: {file_path}")

    log_event(f"\n📁 Total files found: {len(files)}")
    log_event(f"✅ Successfully appended: {len(df_list)}")
    log_event(f"❌ Skipped due to errors: {len(files) - len(df_list)}\n")

    # -----------------------
    # Combine all
    # -----------------------
    if df_list:
        Shopify_ledger_overall = pd.concat(df_list, ignore_index=True)
    else:
        Shopify_ledger_overall = pd.DataFrame(columns=columns_to_read)

    # -----------------------
    # Clean and Format
    # -----------------------
    if not Shopify_ledger_overall.empty:
        Shopify_ledger_overall = Shopify_ledger_overall[~Shopify_ledger_overall['Product variant sku'].isnull()]
        Shopify_ledger_overall['country'] = "UK"
        Shopify_ledger_overall['platform'] = "Shopify"

        Shopify_ledger_overall.rename(
            columns={
                'Product variant sku': 'SKU',
                'Day': 'Date',
                'Ending inventory units': 'Ending_Warehouse_Balance'},
            inplace=True)

        # SKU cleanup
        Shopify_ledger_overall['SKU'] = 'KCA-' + Shopify_ledger_overall['SKU'].astype(str).str.replace(r'\.0$', '', regex=True)
        Shopify_ledger_overall = Shopify_ledger_overall[Shopify_ledger_overall['SKU'] != 'KCA-nan']

        # Create Year-Month for aggregation
        Shopify_ledger_overall['Year-Month'] = Shopify_ledger_overall['Date'].dt.strftime('%Y-%m')

        # Convert Ending_Warehouse_Balance to numeric
        Shopify_ledger_overall['Ending_Warehouse_Balance'] = pd.to_numeric(Shopify_ledger_overall['Ending_Warehouse_Balance'], errors='coerce').fillna(0)